## Experiment 07 (Modified Version): Regularized Feedforward Neural Network for Handwritten Digit Classification

**Course:** ICE 4111: Artificial Intelligence Lab  
**Program:** BSc in Information and Communication Engineering  
**Student Name:** ______________________________  
**Student ID:** ______________________________  
**Section/Batch:** ______________________________  
**Date:** ______________________________  

**Modified experiment:** This version changes the network architecture, regularization, training settings, and analysis plots. It is designed to run in Kaggle Notebook using the built-in Scikit-learn Digits dataset, so no external dataset download is needed.


### Quick Overview

This notebook trains a regularized feedforward neural network on the Scikit-learn Digits dataset. Compared with a basic two-hidden-layer network, this version uses three hidden layers, Batch Normalization, Dropout, a lower learning rate, and a different train/validation/test split seed. It also evaluates prediction confidence and highlights the most difficult digit classes.


### **Learning Objectives**

After completing this experiment, students will be able to:

- Train a feedforward neural network for multiclass classification using backpropagation.
- Apply the Adam optimizer and early stopping to a training loop.
- Monitor training and validation curves to diagnose underfitting and overfitting.
- Evaluate a multiclass model using accuracy, a classification report, and a confusion matrix.
- Visually inspect correctly and incorrectly classified samples to understand model behavior.


## **Theory**

### Introduction

A feedforward neural network passes information from the input layer through one or more hidden layers to the output layer without cycles. Training uses the **backpropagation algorithm** to compute how each weight contributes to the final error, and a gradient-based optimizer then updates the weights to reduce that error. This experiment focuses on the learning process itself — how loss and accuracy evolve over epochs — which is central to all modern deep learning systems.

### Mathematical Background

If the loss function is $L(\hat{y}, y)$ and the network output is obtained by composing layers, backpropagation uses the chain rule to compute gradients such as $\partial L / \partial W$. For a layer $z = Wa + b$ with activation $a' = \phi(z)$, the gradient of the loss is propagated backward from the output layer to earlier layers:

$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial a'} \cdot \frac{\partial a'}{\partial z} \cdot \frac{\partial z}{\partial W}$$

For multiclass classification, the output layer uses a **softmax** activation

$$\hat{y}_k = \frac{e^{z_k}}{\sum_{j} e^{z_j}}$$

so that the outputs form a probability distribution over all classes, and the network is trained with **categorical cross-entropy loss**:

$$L = -\sum_{k=1}^{K} y_k \log(\hat{y}_k)$$

Gradient-based optimizers such as Adam then update the parameters using the computed derivatives, combining momentum with per-parameter adaptive learning rates.

### Important Concepts

The practical training loop is: initialize weights, perform forward propagation, compute the loss, backpropagate gradients, update weights, and repeat for many epochs. **Early stopping** halts training when validation performance no longer improves for a set number of epochs (patience), and restores the weights from the best epoch. In a feedforward network, hidden layers learn intermediate representations that can be more useful for classification than the raw input features alone — this is one reason deep networks generalize better than a single linear layer on many multiclass tasks.

Labels for multiclass problems are converted to **one-hot vectors** (e.g. digit `3` becomes `[0,0,0,1,0,0,0,0,0,0]`) so they can be compared directly against the softmax output using categorical cross-entropy.

### Advantages and Applications

**Advantages**

- Feedforward networks are easy to build and are a strong baseline for classification problems.
- Backpropagation scales efficiently to networks with many layers and parameters.
- Hidden layers learn useful intermediate features automatically, without manual feature engineering.

**Disadvantages**

- Feedforward networks are less effective than specialized architectures (CNNs for images, RNNs/Transformers for sequences) because they ignore spatial or temporal structure.
- Training is sensitive to hyperparameters such as learning rate, layer sizes, and initialization.

**Applications**

Feedforward networks remain important as baselines and as building blocks inside larger systems, and are directly useful for tabular data classification, simple pattern recognition, and — as in this lab — recognizing small, low-resolution images such as handwritten digits.

```mermaid
flowchart LR
    A[Input Layer: 64 Pixels] --> B[Hidden Layer 1: ReLU]
    B --> C[Hidden Layer 2: ReLU]
    C --> D[Output Layer: Softmax]
    D --> E[Categorical Cross-Entropy Loss]
    E --> F[Backpropagation]
    F --> B
    F --> C
```


## **Required Software and Libraries**

Install or prepare the following before starting the lab:

- Python 3.12 or later
- Jupyter Notebook, JupyterLab, VS Code, or Google Colab
- NumPy
- Pandas
- Matplotlib
- TensorFlow 2.x / Keras
- Scikit-learn

### Starter Check

Run the first code cell (library imports and setup). If it prints `Setup complete. You can start the experiment.`, your environment is ready.

## Dataset Description

The experiment uses Scikit-learn's built-in **Digits dataset**.

- **Input:** 8×8 grayscale images flattened into 64 features.
- **Pixel scaling:** original intensities (0–16) are divided by 16 to obtain values in [0, 1].
- **Classes:** handwritten digits 0–9.
- **Kaggle compatibility:** the dataset is included in Scikit-learn, so internet access is not required.


## Experimental Procedure

1. Load and scale the Digits dataset.
2. Split the data into training, validation, and test sets with stratification.
3. Build a three-hidden-layer neural network with Batch Normalization and Dropout.
4. Train using Adam and Early Stopping.
5. Evaluate test accuracy, precision, recall, and F1-score.
6. Inspect learning curves, the confusion matrix, per-class accuracy, prediction confidence, and sample errors.


## Source Code — Modified Version

Run the cells from top to bottom. The code uses standard Kaggle-compatible libraries and does not require uploading a separate dataset. Results may vary slightly with software versions.


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

from sklearn.datasets import load_digits
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score
)
from sklearn.model_selection import train_test_split

# Use a different seed from the original notebook
SEED = 21
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Setup complete — modified experiment is ready.")


In [ ]:
# Load and scale the Digits dataset
digits = load_digits()
X = digits.data.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)

print("Feature matrix shape:", X.shape)
print("Number of classes:", len(np.unique(y)))
print("Samples per class:", np.bincount(y))
print(f"Scaled pixel range: {X.min():.2f} to {X.max():.2f}")


In [ ]:
# Show a varied sample of digits
sample_ids = [7, 28, 51, 83, 109, 147, 202, 251, 300, 351]
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for ax, idx in zip(axes.ravel(), sample_ids):
    ax.imshow(digits.images[idx], cmap="magma")
    ax.set_title(f"Class {y[idx]}")
    ax.axis("off")
plt.suptitle("Handwritten Digit Samples (Modified View)", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Stratified train / validation / test split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.18, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.18,
    random_state=SEED, stratify=y_train_full
)

print(f"Training samples   : {len(X_train)}")
print(f"Validation samples : {len(X_val)}")
print(f"Test samples       : {len(X_test)}")


In [ ]:
# Build a deeper, regularized feedforward network
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(64,)),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.30),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.20),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.10),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0007),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()


In [ ]:
# Train with early stopping and learning-rate reduction
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3,
        min_lr=1e-5, verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=24,
    callbacks=callbacks,
    verbose=1
)

print(f"Training completed in {len(history.history['loss'])} epochs.")


In [ ]:
# Evaluate on held-out test data
test_probabilities = model.predict(X_test, verbose=0)
test_predictions = np.argmax(test_probabilities, axis=1)
test_true = y_test

test_accuracy = accuracy_score(test_true, test_predictions)
macro_f1 = f1_score(test_true, test_predictions, average="macro")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Macro F1-score: {macro_f1:.4f}")
print("\nClassification report:")
print(classification_report(test_true, test_predictions, digits=4, zero_division=0))


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Training")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy by Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Training")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Loss by Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Sparse Cross-Entropy")
axes[1].legend()
fig.suptitle("Modified Network: Learning Curves")
plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrix for the modified model
cm = confusion_matrix(test_true, test_predictions, labels=np.arange(10))
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=np.arange(10)).plot(
    ax=ax, cmap="Purples", colorbar=False, values_format="d"
)
ax.set_title("Test Confusion Matrix — Modified Network")
plt.tight_layout()
plt.show()


In [ ]:
# Per-class accuracy and sample count
class_totals = cm.sum(axis=1)
per_class_accuracy = np.divide(
    cm.diagonal(), class_totals,
    out=np.zeros_like(cm.diagonal(), dtype=float),
    where=class_totals != 0
)
plt.figure(figsize=(9, 4.5))
plt.bar(np.arange(10), per_class_accuracy)
plt.xlabel("True digit class")
plt.ylabel("Accuracy")
plt.title("Per-Class Test Accuracy")
plt.xticks(np.arange(10))
plt.ylim(0, 1.05)
plt.grid(axis="y", linestyle=":", alpha=0.5)
plt.show()

hardest = np.argsort(per_class_accuracy)[:3]
print("Three lowest-accuracy classes:")
for digit in hardest:
    print(f"Digit {digit}: {per_class_accuracy[digit]:.3f} accuracy ({cm[digit].sum()} test samples)")


In [ ]:
# Inspect both correct and incorrect predictions
correct = np.where(test_predictions == test_true)[0]
wrong = np.where(test_predictions != test_true)[0]
print(f"Correct predictions: {len(correct)} / {len(test_true)}")
print(f"Incorrect predictions: {len(wrong)} / {len(test_true)}")

rng = np.random.default_rng(SEED)
chosen_correct = rng.choice(correct, size=min(5, len(correct)), replace=False)
chosen_wrong = wrong[:5]

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for ax, idx in zip(axes[0], chosen_correct):
    ax.imshow(X_test[idx].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"True {test_true[idx]} | Pred {test_predictions[idx]}")
    ax.axis("off")
for ax, idx in zip(axes[1], chosen_wrong):
    confidence = test_probabilities[idx, test_predictions[idx]]
    ax.imshow(X_test[idx].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"True {test_true[idx]} | Pred {test_predictions[idx]}\nConf. {confidence:.2f}")
    ax.axis("off")
for ax in axes[0][len(chosen_correct):]:
    ax.axis("off")
for ax in axes[1][len(chosen_wrong):]:
    ax.axis("off")
axes[0, 0].set_ylabel("Correct")
axes[1, 0].set_ylabel("Incorrect")
plt.suptitle("Example Predictions: Correct vs Incorrect")
plt.tight_layout()
plt.show()


In [ ]:
# Prediction confidence analysis (new output)
confidence = np.max(test_probabilities, axis=1)
correct_mask = test_predictions == test_true

print(f"Mean confidence on correct predictions  : {confidence[correct_mask].mean():.3f}" if correct_mask.any() else "No correct predictions.")
print(f"Mean confidence on incorrect predictions: {confidence[~correct_mask].mean():.3f}" if (~correct_mask).any() else "No incorrect predictions.")

plt.figure(figsize=(8, 4))
plt.hist(confidence[correct_mask], bins=12, alpha=0.7, label="Correct")
if (~correct_mask).any():
    plt.hist(confidence[~correct_mask], bins=12, alpha=0.7, label="Incorrect")
plt.xlabel("Maximum predicted probability")
plt.ylabel("Number of test samples")
plt.title("Model Confidence on Test Predictions")
plt.legend()
plt.tight_layout()
plt.show()


## Expected Output

When the notebook runs successfully, it will display dataset details, digit samples, the split sizes, the model summary, training logs, a classification report, learning curves, a confusion matrix, per-class accuracy, correct/incorrect examples, and a confidence histogram. Exact scores depend on the training environment and should be taken from your own run.


## Performance Evaluation

Use test accuracy and macro F1-score together. Accuracy measures the overall fraction of correct predictions, while macro F1 gives equal weight to each digit class. The confusion matrix and per-class accuracy reveal which digits are difficult for the network. The confidence histogram shows whether the model assigns different confidence levels to correct and incorrect predictions; high confidence does not guarantee that a prediction is correct.


## Viva Questions

1. **What is backpropagation?** It computes gradients of the loss with respect to network weights.
2. **Why use Batch Normalization?** It normalizes intermediate activations and can make training more stable.
3. **What is Dropout?** A regularization method that randomly disables some units during training.
4. **Why use Early Stopping?** It restores the best validation-loss weights and helps reduce overfitting.
5. **Why use sparse categorical cross-entropy here?** Labels are integer class IDs, so one-hot encoding is unnecessary.
6. **What does a confusion matrix show?** It shows counts of correct and incorrect predictions for each class.
7. **What is macro F1-score?** The average F1-score across classes, giving each class equal weight.
8. **Does high confidence mean a prediction is correct?** No. A model can be confidently wrong.


## Exercises and Submission Checklist

1. Compare this network with a simpler 128/64 network.
2. Change Dropout values and compare validation loss.
3. Try batch sizes 16, 32, and 48.
4. Compare test accuracy and macro F1-score.
5. Include your own outputs, plots, and a short explanation in the lab report.

**Submission reminder:** Report the actual metrics produced by your Kaggle run. Do not copy expected or guessed accuracy values.


## **Exercises**

Complete the following tasks after running the notebook:

1. Change the number of hidden units (e.g. 64/32 instead of 128/64) and compare the test accuracy.
2. Try `tanh` activation instead of `relu` in the hidden layers and observe the effect on convergence speed.
3. Add a `Dropout` layer after each hidden layer and check whether validation loss improves or the train/validation gap narrows.
4. Train the network without early stopping (set a high fixed number of epochs) and compare the resulting learning curves.
5. Replace the Digits dataset with another multiclass dataset from Scikit-learn (e.g. `load_wine`) and adapt the input/output layer sizes accordingly.

## **Lab Report Submission Checklist**

Before submitting, confirm that your report includes:

- Completed cover information.
- Objective and theory summary in your own words.
- All important code outputs, including the classification report.
- The learning curves, confusion matrix, per-class accuracy chart, and sample prediction images.
- Short discussion of which digits were most often confused and why.
- Answers to assigned viva questions.
- Completed exercises.

**Reminder:** A good lab report should not only show screenshots. It should explain what the outputs mean.
